In [ ]:
import pandas as pd
from openai import OpenAI

from subprocess import check_output

In [ ]:
DESCRIPTION_COLUMN_NAME = "Description (Islandora)"

MODELS = ["gpt-oss", "minimax-m2"]

In [ ]:
class SubjectExtractor(BaseModel):
    subjects: list[str]

class GeographyExtractor(BaseModel):
    location: list[str]

In [ ]:
EXTRACTORS = [
    {"name": "subject", "class": SubjectExtractor, "prompt_path": "../prompts/Extract_Subject.md"},
]

In [ ]:
df = pd.read_excel("../data/input/KerrMcGeeMetadataSetAITest.xlsx")

In [ ]:
descriptions = df[DESCRIPTION_COLUMN_NAME]

In [ ]:
descriptions

In [ ]:
with open("../prompts/Extract_Subject.md", "r") as f:
    systemprompt = f.read()

In [ ]:
def extract(model: str, system_prompt: str, extractor: type[BaseModel], record_content: str): 
    response = client.responses.parse(
        model=model,
        input=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {"role": "user", "content": record_content},
        ],
        text_format=extractor,
    )

    return response.output_parsed

In [ ]:
extract("gpt-oss", systemprompt, SubjectExtractor, descriptions[0])

In [8]:
for model in MODELS:
    for extractor in EXTRACTORS:
        tmp_records = []
        for record in descriptions:
            with open(extractor["prompt_path"], "r") as f:
                systemprompt = f.read()
            tmp_records.append(extract(model, systemprompt, extractor["class"], record))
        df[f"{extractor["name"]}_{model}"] = tmp_records
        print(tmp_records)

df.to_csv("../data/results.csv", index=False)
        

NameError: name 'MODELS' is not defined

In [2]:
from utils import file_to_df, read_prompt, extract_multi
from pydantic import BaseModel

In [10]:
loaded_data = file_to_df("../data/input/KerrMcGeeMetadataSetAITest.xlsx")
system_prompt = read_prompt("Extract_Subject")

In [11]:
descriptions = loaded_data["Description (Islandora)"]

In [12]:
class SubjectExtractor(BaseModel):
    subjects: list[str]

results = extract_multi("gpt-oss", system_prompt, SubjectExtractor, list(descriptions))

In [16]:
next(results)

SubjectExtractor(subjects=['oil and gas industry', 'drilling rigs', 'industrial photography'])

In [22]:
from utils import file_to_df, read_prompt, extract_multi
from pydantic import BaseModel

class SubjectExtractor(BaseModel):
    subjects: list[str]

loaded_data = file_to_df("../data/input/KerrMcGeeMetadataSetAITest.xlsx")
    
results = extract_multi(
    model="anthropic.claude-3-haiku-20240307-v1:0",
    system_prompt=read_prompt("Extract_Subject"),
    extractor=SubjectExtractor,
    record_contents=loaded_data["Description (Islandora)"]
)

haiku = results

In [23]:
results = extract_multi(
    model="gpt-oss",
    system_prompt=read_prompt("Extract_Subject"),
    extractor=SubjectExtractor,
    record_contents=loaded_data["Description (Islandora)"]
)

gpt_oss = results

In [11]:
loaded_data["Description (Islandora)"][0]

"Black and white photograph depicting the Kerr Inn Service gas station in Oklahoma City, Oklahoma described as Lou Lindsay's Gas Station, looking southeast. The service station was located on the northeast corner of N Walker Ave. and Couch Dr. in Oklahoma City."

In [31]:
result1 = set(next(haiku).subjects)
result2 = set(next(gpt_oss).subjects)

In [32]:
print(result1)
print(result2)
print(result1 | result2)  # outer join 
print(result1 & result2)  # inner join

{'oil wells', 'drilling rigs', 'industrial photography', 'petroleum industry'}
{'drilling rigs', 'industrial photography', 'oil and gas industry'}
{'oil wells', 'drilling rigs', 'industrial photography', 'oil and gas industry', 'petroleum industry'}
{'drilling rigs', 'industrial photography'}
